# Phase 1: Data Loading & Exploration

**Goal:** Load all datasets, understand their shape, columns, and what each row represents.

**Key question to answer by end of this notebook:**
- How many total IPL matches are in the dataset?
- Which team has won the most matches?
- What % of matches does the toss winner also win?
- Which venue has hosted the most matches?

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Show all columns when printing a dataframe
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Libraries loaded successfully')

## 2. Load the Ball-by-Ball Data

**What this file is:** Every single delivery bowled in IPL 2008–2025.
Each row = one ball. One match has ~240–300 rows (120 balls per innings × 2 innings).
It also contains match-level info (toss, venue, winner) repeated on every row.

In [ ]:
# Load the ball-by-ball data
# parse_dates tells pandas to read the 'date' column as a proper date, not a string
ball_df = pd.read_csv('../data/raw/matches.csv', parse_dates=['date'])

print(f'Shape: {ball_df.shape}')   # (rows, columns)
print(f'Rows: {ball_df.shape[0]:,}')   # total deliveries
print(f'Columns: {ball_df.shape[1]}')

In [ ]:
# See the first 3 rows
# Each row is ONE ball bowled in a match
ball_df.head(3)

In [ ]:
# See all column names and their data types
# 'object' = text/string, 'int64'/'float64' = numbers, 'datetime64' = dates
ball_df.info()

In [ ]:
# Count null (missing) values per column
# Columns with many nulls need special handling in Phase 2
null_counts = ball_df.isnull().sum()
null_counts[null_counts > 0].sort_values(ascending=False)

## 3. Extract Match-Level Data

The ball-by-ball file has match info (toss, venue, winner) repeated on every ball.
We deduplicate to get ONE row per match — this is our matches table.

In [ ]:
# Columns that are the same for every ball in a match (match-level info)
MATCH_COLS = [
    'match_id', 'date', 'season', 'venue', 'city',
    'batting_team', 'bowling_team',
    'toss_winner', 'toss_decision',
    'match_won_by', 'win_outcome',
    'stage', 'event_match_no'
]

# Keep only columns that exist in our data
available_cols = [c for c in MATCH_COLS if c in ball_df.columns]

# drop_duplicates: keep only the first row per match_id
# This collapses hundreds of ball rows into one match row
matches_df = (
    ball_df[available_cols]
    .drop_duplicates(subset='match_id')
    .reset_index(drop=True)
)

print(f'Total unique matches: {len(matches_df):,}')
matches_df.head()

In [ ]:
# What seasons are in the data?
# value_counts() counts how many matches per season
# The season column has mixed formats: '2007/08' (old) and 2023 (new integer)
# Convert everything to string first so sort_index() can compare apples to apples
print('Matches per season:')
print(matches_df['season'].astype(str).value_counts().sort_index())

In [ ]:
# What does match_won_by look like?
# This column tells us WHO won — we need to understand its format
print('Sample match_won_by values:')
print(matches_df['match_won_by'].value_counts().head(20))

In [ ]:
# Extract the winning team from match_won_by
# The column format is e.g. "Mumbai Indians" or "140 runs" (which means batting team won)
# The batting_team in the FIRST innings is the one who set the target
# We need to figure out winner from this

# Let's check the unique values to understand the format
print('Unique win_outcome values:')
print(matches_df['win_outcome'].value_counts().head(10) if 'win_outcome' in matches_df.columns else 'column not found')

In [ ]:
# Get the winner for each match
# match_won_by contains the winning team name (e.g. 'Mumbai Indians')
# Let's verify this by checking a few matches

# First innings batting team = team that batted first
# Let's cross-check: does match_won_by match a known team name?
all_teams = pd.concat([
    ball_df['batting_team'],
    ball_df['bowling_team']
]).unique()

print(f'Total unique team names found: {len(all_teams)}')
print(sorted(all_teams))

## 4. Key Match Statistics

In [ ]:
# Which team has won the most matches?
# value_counts() on match_won_by gives wins per team
print('Top 10 teams by match wins:')
print(matches_df['match_won_by'].value_counts().head(10))

In [ ]:
# Which venue has hosted the most matches?
print('Top 10 venues by matches hosted:')
print(matches_df['venue'].value_counts().head(10))

In [ ]:
# Toss analysis — does winning the toss help?
# We check: in how many matches did the toss winner also win the match?
toss_win = matches_df[
    matches_df['toss_winner'] == matches_df['match_won_by']
]
toss_win_pct = len(toss_win) / len(matches_df) * 100
print(f'Matches where toss winner also won: {len(toss_win)}')
print(f'Total matches: {len(matches_df)}')
print(f'Toss win % = {toss_win_pct:.1f}%')

In [ ]:
# Toss decision breakdown — do teams prefer to bat or field?
print('Toss decisions (bat vs field):')
print(matches_df['toss_decision'].value_counts())

## 5. Ball-by-Ball Summary

In [ ]:
# Total deliveries in dataset
print(f'Total deliveries: {len(ball_df):,}')

# Unique batters
print(f'Unique batters: {ball_df["batter"].nunique()}')

# Unique bowlers
print(f'Unique bowlers: {ball_df["bowler"].nunique()}')

# Date range
print(f'Date range: {ball_df["date"].min()} → {ball_df["date"].max()}')

In [ ]:
# Top 10 run scorers (career total from ball-by-ball)
# groupby batter, sum up their runs_batter
top_batters = (
    ball_df.groupby('batter')['runs_batter']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print('Top 10 run scorers in dataset:')
print(top_batters)

In [ ]:
# Top 10 wicket takers (career total from ball-by-ball)
# A wicket row has a non-null wicket_kind AND it's not a run-out (bowler doesn't get credit)
# For simplicity, count all dismissals credited to bowler
bowler_wickets = (
    ball_df[ball_df['bowler_wicket'] == 1]
    .groupby('bowler')['bowler_wicket']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print('Top 10 wicket takers:')
print(bowler_wickets)

## 6. Load Player Stats

In [ ]:
# Load the pre-computed player stats CSV
player_df = pd.read_csv('../data/raw/player_stats.csv')

print(f'Shape: {player_df.shape}')
player_df.head(3)

In [ ]:
# What columns do we have?
print(player_df.columns.tolist())

In [ ]:
# Check for nulls in player stats
null_counts_player = player_df.isnull().sum()
null_counts_player[null_counts_player > 0]

## 7. Load IPL 2026 Squad Data

In [ ]:
# Load 2026 auction / squad data
squads_df = pd.read_csv('../data/raw/ipl_2026_squads.csv')

print(f'Shape: {squads_df.shape}')
squads_df.head(5)

In [ ]:
# What columns do we have?
print(squads_df.columns.tolist())

# How many players per team?
print('\nPlayers per team:')
team_col = 'TeamName' if 'TeamName' in squads_df.columns else squads_df.columns[5]
print(squads_df[team_col].value_counts())

## 8. Summary — What We Have

Run this cell after completing the notebook to confirm your understanding.

In [ ]:
print('=' * 50)
print('DATASET SUMMARY')
print('=' * 50)
print(f'Ball-by-ball deliveries : {len(ball_df):>10,}')
print(f'Unique matches          : {ball_df["match_id"].nunique():>10,}')
print(f'Seasons covered         : {ball_df["season"].nunique():>10}')
print(f'Unique batters          : {ball_df["batter"].nunique():>10}')
print(f'Unique bowlers          : {ball_df["bowler"].nunique():>10}')
print(f'Player stats rows       : {len(player_df):>10,}')
print(f'2026 squad players      : {len(squads_df):>10,}')
print('=' * 50)

## Your Tasks (Do These Before Moving to Phase 2)

```
□ Run every cell top to bottom — no errors
□ Write down answers to these 4 questions (no peeking at code):
  - How many total IPL matches are in the dataset?
  - Which team has won the most matches?
  - What % of matches does the toss winner also win?
  - Which venue has hosted the most matches?

□ For each of these functions, write ONE sentence explaining what it does:
  - pd.read_csv()
  - df.shape
  - df.isnull().sum()
  - df.drop_duplicates(subset='match_id')
  - df.groupby('batter')['runs_batter'].sum()
  - df.value_counts()

□ Add to your interview notebook:
  Q: "What's the first thing you do when you get a new dataset?"
  A: [write your answer based on what you just did]
```